In [1]:
from models.rnn_decoder import RNNDecoder
import os
from omegaconf import OmegaConf

model_path = 'trained_models/baseline_lstm_bi'
model_args = OmegaConf.load(os.path.join(model_path, 'checkpoint/args.yaml'))

model = RNNDecoder(
            neuron_capture_tensor_dim = model_args['model']['n_input_features'],
            hidden_state_dim = model_args['model']['n_units'],
            num_days = len(model_args['dataset']['sessions']),
            num_phonemes = model_args['dataset']['n_classes'],
            rnn_type = model_args['model']['rnn_type'],
            rnn_dropout = model_args['model']['rnn_dropout'],
            input_dropout = model_args['model']['input_network']['input_layer_dropout'],
            num_rec_layers = model_args['model']['n_layers'],
            ts_patch_size = model_args['model']['patch_size'],
            ts_patch_stride = model_args['model']['patch_stride'],
            bidirectional = model_args['model']['bidirectional']
        )

In [2]:
import torch

device = torch.device('mps') # torch.device(f'cuda:0')
checkpoint = torch.load(os.path.join(model_path, 'checkpoint/best_checkpoint'),
                          weights_only=False, map_location=device)

In [3]:
for key in list(checkpoint['model_state_dict'].keys()):
    checkpoint['model_state_dict'][key.replace("module.", "")] = checkpoint['model_state_dict'].pop(key)
    checkpoint['model_state_dict'][key.replace("_orig_mod.", "")] = checkpoint['model_state_dict'].pop(key)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

RNNDecoder(
  (day_weights): ParameterList(
      (0): Parameter containing: [torch.float32 of size 512x512]
      (1): Parameter containing: [torch.float32 of size 512x512]
      (2): Parameter containing: [torch.float32 of size 512x512]
      (3): Parameter containing: [torch.float32 of size 512x512]
      (4): Parameter containing: [torch.float32 of size 512x512]
      (5): Parameter containing: [torch.float32 of size 512x512]
      (6): Parameter containing: [torch.float32 of size 512x512]
      (7): Parameter containing: [torch.float32 of size 512x512]
      (8): Parameter containing: [torch.float32 of size 512x512]
      (9): Parameter containing: [torch.float32 of size 512x512]
      (10): Parameter containing: [torch.float32 of size 512x512]
      (11): Parameter containing: [torch.float32 of size 512x512]
      (12): Parameter containing: [torch.float32 of size 512x512]
      (13): Parameter containing: [torch.float32 of size 512x512]
      (14): Parameter containing: [torch.f

In [4]:
import pandas as pd
from utils.load_data import load_h5py_file

eval_type = 'val'
csv_path = '../data/t15_copyTaskData_description.csv'
b2txt_csv_df = pd.read_csv(csv_path)
data_dir = "../data/hdf5_data_final"

print(f"Loading {eval_type} data...")

test_data = {}
total_test_trials = 0

for session in model_args['dataset']['sessions']:
    eval_file = os.path.join(data_dir, session, f'data_{eval_type}.hdf5')

    if os.path.exists(eval_file):
        data = load_h5py_file(eval_file, b2txt_csv_df)
        test_data[session] = data
        total_test_trials += len(data["neural_features"])
        print(f'Loaded {len(data["neural_features"])} trials for {session}')

print(f'Total {eval_type} trials: {total_test_trials}\n')

Loading val data...
Loaded 35 trials for t15.2023.08.13
Loaded 49 trials for t15.2023.08.18
Loaded 48 trials for t15.2023.08.20
Loaded 25 trials for t15.2023.08.25
Loaded 25 trials for t15.2023.08.27
Loaded 49 trials for t15.2023.09.01
Loaded 34 trials for t15.2023.09.03
Loaded 35 trials for t15.2023.09.24
Loaded 48 trials for t15.2023.09.29
Loaded 44 trials for t15.2023.10.01
Loaded 36 trials for t15.2023.10.06
Loaded 17 trials for t15.2023.10.08
Loaded 44 trials for t15.2023.10.13
Loaded 44 trials for t15.2023.10.15
Loaded 9 trials for t15.2023.10.20
Loaded 33 trials for t15.2023.10.22
Loaded 50 trials for t15.2023.11.03
Loaded 15 trials for t15.2023.11.04
Loaded 25 trials for t15.2023.11.17
Loaded 20 trials for t15.2023.11.19
Loaded 44 trials for t15.2023.11.26
Loaded 34 trials for t15.2023.12.03
Loaded 50 trials for t15.2023.12.08
Loaded 25 trials for t15.2023.12.10
Loaded 30 trials for t15.2023.12.17
Loaded 50 trials for t15.2023.12.29
Loaded 23 trials for t15.2024.02.25
Loaded 24

In [ ]:
from models.data_augmentations import gauss_smooth
from tqdm import tqdm
import numpy as np

def runSingleDecodingStep(x, input_layer, model, model_args, device):

    # Use autocast for efficiency
    with torch.autocast(device_type = device, enabled = model_args['use_amp'], dtype = torch.bfloat16):

        x = gauss_smooth(
            inputs = x, 
            device = device,
            smooth_kernel_std = model_args['dataset']['data_transforms']['smooth_kernel_std'],
            smooth_kernel_size = model_args['dataset']['data_transforms']['smooth_kernel_size'],
            padding = 'valid',
        )

        with torch.no_grad():
            logits, _ = model(
                x = x,
                day_idx = torch.tensor([input_layer], device=device),
                states = None,
                return_state = True,
            )

    logits = logits.float().cpu().numpy()

    return logits

with tqdm(total=total_test_trials, desc='Predicting phoneme sequences', unit='trial') as pbar:
    for session, data in test_data.items():

        data['logits'] = []
        data['pred_seq'] = []
        input_layer = model_args['dataset']['sessions'].index(session)
        
        for trial in range(len(data['neural_features'])):
            # get neural input for the trial
            neural_input = data['neural_features'][trial]

            # add batch dimension
            neural_input = np.expand_dims(neural_input, axis=0)

            # convert to torch tensor
            neural_input = torch.tensor(neural_input, device=device, dtype=torch.bfloat16)

            # run decoding step
            logits = runSingleDecodingStep(neural_input, input_layer, model, model_args, device.__str__())
            data['logits'].append(logits)

            pbar.update(1)
pbar.close()

Predicting phoneme sequences:   0%|          | 0/1426 [00:00<?, ?trial/s]

In [6]:
import importlib

# Assuming 'my_module' is the module you want to reload
import simple_phoneme_to_text

importlib.reload(simple_phoneme_to_text)

<module 'simple_phoneme_to_text' from '/Users/rishabhsood/Desktop/Personal/OMSCS/2025/FA25/CS7643_Deep-Learning/Group Project/BrainToText25/github/brain-to-text-25/models/simple_phoneme_to_text.py'>

In [12]:
from simple_phoneme_to_text import SimplePhonemeToTextConverter

converter = SimplePhonemeToTextConverter(
        dictionary_path=None,
        use_ngrams=False
    )
use_beam_search = False

with tqdm(total=total_test_trials, desc='Predicting phoneme sequences', unit='trial') as pbar:
    for session, data in test_data.items():
        data['pred_phonemes'] = []

        for trial in range(len(data['logits'])):
            logits = data['logits'][trial][0]  # Remove batch dim

            # Decode using your converter
            if use_beam_search:
                phonemes = converter.decode_ctc_beam_search(logits, beam_width=10)
            else:
                phonemes = converter.decode_ctc_greedy(logits)

            data['pred_phonemes'].append(phonemes)
            pbar.update(1)
pbar.close()

Predicting phoneme sequences: 100%|██████████| 1426/1426 [00:00<00:00, 23741.78trial/s]


In [13]:
from simple_phoneme_to_text import LOGIT_TO_PHONEME

for session, data in test_data.items():
    true_phonemes = []
    for seq in data['seq_class_ids']:
        true_phoneme_seq = []
        for pid in seq: 
            if pid == 0: break
            true_phoneme_seq.append(LOGIT_TO_PHONEME[pid])
        true_phonemes.append(true_phoneme_seq)
    data['true_phonemes'] = true_phonemes

In [14]:
data.keys()

dict_keys(['neural_features', 'n_time_steps', 'seq_class_ids', 'seq_len', 'transcriptions', 'sentence_label', 'session', 'block_num', 'trial_num', 'corpus', 'logits', 'pred_seq', 'pred_phonemes', 'true_phonemes'])

In [15]:
results = []

for session, data in test_data.items():
    results.append(
        {
            'sentence_label': data['sentence_label'],
            'pred_phonemes': data['pred_phonemes'], 
            'true_phonemes': data['true_phonemes'], 
            'session': data['session'], 
            'block_num': data['block_num'], 
            'trial_num': data['trial_num']
        }
    )

In [16]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv('phoneme_prediction_results_greedy.csv')
# df.head()

In [17]:
df.head()

,sentence_label,pred_phonemes,true_phonemes,session,block_num,trial_num
0,"[You can see the code at this point as well., ...","[[ | , OW, | , AY, T, | , OW, L], [AW, | , ...","[[Y, UW, | , K, AE, N, | , S, IY, | , DH, A...","[t15.2023.08.13, t15.2023.08.13, t15.2023.08.1...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1,"[They recently released him., One year public ...","[[EH, AH, L, IH, T, | ], [W, IY, | , P, L, S...","[[DH, EY, | , R, IY, S, AH, N, T, L, IY, | ,...","[t15.2023.08.18, t15.2023.08.18, t15.2023.08.1...","[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
2,"[Do you go by the ads when you look at them?, ...","[[UW, | , Y, UW, L, | , Y, UW, | , Y, | ],...","[[D, UW, | , Y, UW, | , G, OW, | , B, AY, ...","[t15.2023.08.20, t15.2023.08.20, t15.2023.08.2...","[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
3,"[Lawmakers passed a measure last year., I'd be...","[[W, L, IH, L, AH, S, T, | , V, R, HH, IY, V,...","[[L, AO, M, EY, K, ER, Z, | , P, AE, S, T, |...","[t15.2023.08.25, t15.2023.08.25, t15.2023.08.2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
4,"[You look down at your arm., The Bermuda Trian...","[[ | , L, AE], [M, AE, N], [], [IH, | , OW, ...","[[Y, UW, | , L, UH, K, | , D, AW, N, | , AE...","[t15.2023.08.27, t15.2023.08.27, t15.2023.08.2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."


## Misc

In [2]:
from dataset import BrainToTextDataset, train_test_split_indicies
from torch.utils.data import DataLoader
from omegaconf import OmegaConf
import os

args = OmegaConf.load('args.yaml')

feature_subset = None
val_file_paths = [os.path.join(args["dataset"]["dataset_dir"],s,'data_val.hdf5') for s in args['dataset']['sessions']]
_, val_trials = train_test_split_indicies(
            file_paths = val_file_paths,
            test_percentage = 1,
            seed = args['dataset']['seed'],
            bad_trials_dict = None,
            )

val_dataset = BrainToTextDataset(
    trial_indicies = val_trials,
    split = 'test',
    days_per_batch = None,
    n_batches = None,
    batch_size = args['dataset']['batch_size'],
    must_include_days = None,
    random_seed = args['dataset']['seed'],
    feature_subset = feature_subset
    )
val_loader = DataLoader(
    val_dataset,
    batch_size = None, # Dataset.__getitem__() already returns batches
    shuffle = False,
    num_workers = 0,
    pin_memory = True
)

In [6]:
for i, batch in enumerate(val_loader):
    print(batch['day_indicies'])

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])


tensor([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2])
tensor([3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
        3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3])
tensor([4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4])
tensor([5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
        5])
tensor([6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
        6])
tensor([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7])
tensor([8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8,
        8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8])
tensor([9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9,

In [1]:
from ctc_beam_search import CTCBeamSearchDecoder
import numpy as np

decoder = CTCBeamSearchDecoder()

In [2]:
# Test 1: log_sum_exp(0, 0) should give log(2) ≈ 0.693
result = decoder._log_sum_exp(0.0, 0.0)
print(f"log_sum_exp(0, 0) = {result}, expected ≈ 0.693")

# Test 2: With -inf
result = decoder._log_sum_exp(-np.inf, 0.0)
print(f"log_sum_exp(-inf, 0) = {result}, expected = 0.0")

# Test 3: Both -inf
result = decoder._log_sum_exp(-np.inf, -np.inf)
print(f"log_sum_exp(-inf, -inf) = {result}, expected = -inf")

# Test 4: From your hand calculation
result = decoder._log_sum_exp(-0.868, -2.120)
print(f"log_sum_exp(-0.868, -2.120) = {result}, expected ≈ -0.617")

log_sum_exp(0, 0) = 0.6931471805599453, expected ≈ 0.693
log_sum_exp(-inf, 0) = 0.0, expected = 0.0
log_sum_exp(-inf, -inf) = -inf, expected = -inf
log_sum_exp(-0.868, -2.120) = -0.6165159728507134, expected ≈ -0.617


In [3]:
beams = {
    (): (-0.5, -1.0),
    (1,): (-0.1, -0.2),
    (2,): (-2.0, -3.0),
    (1, 2): (-0.3, -0.4)
}

decoder._prune_beams(beams, 2)

{(1,): (-0.1, -0.2), (1, 2): (-0.3, -0.4)}

In [5]:
for (k, v) in beams.items():
    print(k, v)

() (-0.5, -1.0)
(1,) (-0.1, -0.2)
(2,) (-2.0, -3.0)
(1, 2) (-0.3, -0.4)


In [6]:
prefix = ()
len(prefix)

0

In [11]:
np = prefix + (1,)
np[-1]

1

In [1]:
import numpy as np
import torch
from ctc_beam_search import CTCBeamSearchDecoder

print("=" * 60)
print("CTC Beam Search Decoder - Test Suite")
print("=" * 60)

# Test 1: Simple sequence (from assignment)
print("\n[Test 1] Simple sequence - should decode to [1]")
log_probs = np.array([
    [-0.5, -1.0, -2.0],  # blank most likely
    [-2.0, -0.1, -2.0],  # 1 most likely
    [-0.5, -1.0, -2.0],  # blank most likely
])
decoder = CTCBeamSearchDecoder(blank_id=0, beam_width=5)
result = decoder.decode_single(log_probs)
print(f"Result: {result}")
print(f"Expected: [1]")
print(f"✓ PASS" if len(result) == 1 and result[0] == 1 else "✗ FAIL")

# Test 2: Empty sequence (all blanks)
print("\n[Test 2] All blanks - should decode to []")
log_probs = np.array([
    [-0.1, -3.0, -3.0],  # blank most likely
    [-0.1, -3.0, -3.0],  # blank most likely
    [-0.1, -3.0, -3.0],  # blank most likely
])
decoder = CTCBeamSearchDecoder(blank_id=0, beam_width=5)
result = decoder.decode_single(log_probs)
print(f"Result: {result}")
print(f"Expected: []")
print(f"✓ PASS" if len(result) == 0 else "✗ FAIL")

# Test 3: Repeated character (need blank separator for "AA")
print("\n[Test 3] Repeated character - 'AA' vs 'A'")
# Path: A, blank, A -> "AA"
log_probs_AA = np.array([
    [-2.0, -0.1, -3.0],  # A most likely
    [-0.1, -2.0, -3.0],  # blank most likely
    [-2.0, -0.1, -3.0],  # A most likely
])
# Path: A, A, A -> "A" (collapses)
log_probs_A = np.array([
    [-2.0, -0.1, -3.0],  # A most likely
    [-2.0, -0.1, -3.0],  # A most likely
    [-2.0, -0.1, -3.0],  # A most likely
])

decoder = CTCBeamSearchDecoder(blank_id=0, beam_width=5)
result_AA = decoder.decode_single(log_probs_AA)
result_A = decoder.decode_single(log_probs_A)

print(f"Result AA: {result_AA} (Expected: [1, 1])")
print(f"Result A:  {result_A} (Expected: [1])")
print(f"✓ PASS" if len(result_AA) == 2 and result_AA[0] == 1 and result_AA[1] == 1 else "✗ FAIL (AA)")
print(f"✓ PASS" if len(result_A) == 1 and result_A[0] == 1 else "✗ FAIL (A)")

# Test 4: Longer sequence "AB"
print("\n[Test 4] Two different characters - should decode to [1, 2] = 'AB'")
log_probs = np.array([
    [-2.0, -0.1, -3.0],  # A
    [-0.5, -2.0, -0.5],  # blank or B (similar probs)
    [-3.0, -3.0, -0.1],  # B
])
decoder = CTCBeamSearchDecoder(blank_id=0, beam_width=5)
result = decoder.decode_single(log_probs)
print(f"Result: {result}")
print(f"Expected: [1, 2]")
print(f"✓ PASS" if len(result) == 2 and result[0] == 1 and result[1] == 2 else "✗ FAIL")

# Test 5: Beam width = 1 (should behave like greedy)
print("\n[Test 5] Beam width = 1 (greedy) vs beam width = 10")
log_probs = np.array([
    [-0.7, -0.8, -2.0],  # blank slightly better
    [-2.0, -0.3, -2.0],  # A much better
    [-0.7, -0.8, -2.0],  # blank slightly better
    [-2.0, -0.3, -2.0],  # A much better
])

decoder_greedy = CTCBeamSearchDecoder(blank_id=0, beam_width=1)
decoder_beam = CTCBeamSearchDecoder(blank_id=0, beam_width=10)

result_greedy = decoder_greedy.decode_single(log_probs)
result_beam = decoder_beam.decode_single(log_probs)

print(f"Greedy (beam=1): {result_greedy}")
print(f"Beam search (beam=10): {result_beam}")
print("Both should give similar results for this simple case")

# Test 6: Batch decoding
print("\n[Test 6] Batch decoding")
batch_logits = torch.randn(3, 5, 4)  # 3 sequences, max_len=5, vocab=4
batch_lengths = torch.tensor([3, 4, 5])  # Different lengths

decoder = CTCBeamSearchDecoder(blank_id=0, beam_width=5)
results = decoder.decode_batch(batch_logits, batch_lengths)

print(f"Decoded {len(results)} sequences")
for i, seq in enumerate(results):
    print(f"  Sequence {i}: {seq} (length: {len(seq)})")
print(f"✓ PASS" if len(results) == 3 else "✗ FAIL")

# Test 7: Your hand-traced example from Task 1.2
print("\n[Test 7] Hand-traced example from Task 1.2")
probs = np.array([
    [0.7, 0.2, 0.1],  # t=0: blank, A, B
    [0.1, 0.6, 0.3],  # t=1
    [0.2, 0.7, 0.1],  # t=2
])
log_probs = np.log(probs)

decoder = CTCBeamSearchDecoder(blank_id=0, beam_width=2)
result = decoder.decode_single(log_probs)
print(f"Result: {result}")
print(f"Expected: [1] (decoded as 'A')")
print(f"✓ PASS" if len(result) == 1 and result[0] == 1 else "✗ FAIL")

print("\n" + "=" * 60)
print("Testing complete!")
print("=" * 60)

CTC Beam Search Decoder - Test Suite

[Test 1] Simple sequence - should decode to [1]
Result: [1]
Expected: [1]
✓ PASS

[Test 2] All blanks - should decode to []
Result: []
Expected: []
✓ PASS

[Test 3] Repeated character - 'AA' vs 'A'
Result AA: [1 1] (Expected: [1, 1])
Result A:  [1] (Expected: [1])
✓ PASS
✓ PASS

[Test 4] Two different characters - should decode to [1, 2] = 'AB'
Result: [1 2]
Expected: [1, 2]
✓ PASS

[Test 5] Beam width = 1 (greedy) vs beam width = 10
Greedy (beam=1): [1]
Beam search (beam=10): [1]
Both should give similar results for this simple case

[Test 6] Batch decoding
Decoded 3 sequences
  Sequence 0: [3] (length: 1)
  Sequence 1: [3 1] (length: 2)
  Sequence 2: [1 2 2] (length: 3)
✓ PASS

[Test 7] Hand-traced example from Task 1.2
Result: [1]
Expected: [1] (decoded as 'A')
✓ PASS

Testing complete!


In [3]:
import numpy as np
arr = np.array([1, -2, 3, -4, 5])
active_tokens = np.where(arr >= -1)

In [4]:
active_tokens

(array([0, 2, 4]),)